## Middleware

#### Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for following:
1. Tracking agent behavior with logging, analytics and debugging.
2. Transforming prompts, tool selection, and output formating.
3. Adding retries, fallbacks, and early termination logic.
4. Applying rate limits, guardrails, and PII detection.

In [12]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')
os.environ['COHERE_API_KEY'] = os.getenv('COHERE_API_KEY')

In [20]:
from langchain.chat_models import init_chat_model

llm = init_chat_model('gemini-2.5-flash-lite', model_provider='google_genai')
summary_model = init_chat_model('command-a-03-2025',model_provider='cohere')
summary_model.invoke('Hello')

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'id': '6142957b-31a0-45f0-a972-64c1b4a844cf', 'finish_reason': 'COMPLETE', 'content': 'Hello! How can I assist you today?', 'token_count': {'input_tokens': 496.0, 'output_tokens': 11.0}}, response_metadata={'id': '6142957b-31a0-45f0-a972-64c1b4a844cf', 'finish_reason': 'COMPLETE', 'content': 'Hello! How can I assist you today?', 'token_count': {'input_tokens': 496.0, 'output_tokens': 11.0}}, id='lc_run--019c0ea2-c344-70c1-80df-0d7b73055447-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 496, 'output_tokens': 11, 'total_tokens': 507})

### Summarization Middleware

#### Automatically summarize conversation history when approaching token limits, preserving recent messages, while compressing older context. Summarization is useful for the following:
- Long running conversations that exceed context windows.
- Multi turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

agent = create_agent(
    model = summary_model,
    middleware = [
        SummarizationMiddleware(model=summary_model,trigger=('messages',10),keep=('messages',4))
    ],
    checkpointer=InMemorySaver()
)

In [22]:
config = {'configurable':{'thread_id':'test-1'}}

questions = [
    'What is 2+2?',
    'What is 10*5?',
    'What is 100/4?',
    'What is 15-7?',
    'What is 3*3?',
    'What is 4*4?'
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='55f359a9-ab52-46ee-be83-0235562b98d5'), AIMessage(content='The answer to 2+2 is 4.', additional_kwargs={'id': '74a639a2-17a9-4020-9bb4-34f1a04838b5', 'finish_reason': 'COMPLETE', 'content': 'The answer to 2+2 is 4.', 'token_count': {'input_tokens': 502.0, 'output_tokens': 13.0}}, response_metadata={'id': '74a639a2-17a9-4020-9bb4-34f1a04838b5', 'finish_reason': 'COMPLETE', 'content': 'The answer to 2+2 is 4.', 'token_count': {'input_tokens': 502.0, 'output_tokens': 13.0}}, id='lc_run--019c0ea3-0e49-7953-abfc-3e7b5cac5dad-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 502, 'output_tokens': 13, 'total_tokens': 515})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='55f359a9-ab52-46ee-be83-0235562b98d5'), AIMessage(content='The answer to 2+2 is 4.', additional_kwargs={'id': '74a639a2-

In [23]:
response

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is seeking assistance with basic arithmetic calculations.\n\n## SUMMARY\nThe user has requested the results of several arithmetic operations: 2+2=4, 10*5=50, 100/4=25. The user is currently awaiting the result of 15-7.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nProvide the user with the result of 15-7.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='4b685c18-1638-4d63-ad92-a8bb4dde68af'),
  AIMessage(content='The answer to 15-7 is 8.', additional_kwargs={'id': '3b105b37-cc85-4041-a608-f92949da3560', 'finish_reason': 'COMPLETE', 'content': 'The answer to 15-7 is 8.', 'token_count': {'input_tokens': 589.0, 'output_tokens': 14.0}}, response_metadata={'id': '3b105b37-cc85-4041-a608-f92949da3560', 'finish_reason': 'COMPLETE', 'content': 'The answer to 15-7 is 8.', 'token_count': {'input_tokens': 589.0, 'output_tokens': 14.0}}, id='lc_run--019c0ea3-1710

## Human in the loop Middleware

#### Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following
- High-stakes operations requiring human approval (e.g., database writes, financial transactions)
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [24]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str) -> str:
    """ Mock function to read an email by its id """
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    ''' Mock function to send an email '''
    return f"Email sent to {recipient} with subject '{subject}'"

In [25]:
agent = create_agent(
    model = summary_model,
    tools = [read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{"allowed_decisions":["approve","edit","reject"]},
                "read_email_tool": False
            }
        )
    ]
)

In [26]:
config = {"configurable":{"thread_id":"test_approve"}}

# Step 1: Request
response = agent.invoke({'messages':[
    HumanMessage(content="Send email to john@test.com with subject 'More information required' and body 'Please call me on whatsapp'")]},
    config=config
)
response

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'More information required' and body 'Please call me on whatsapp'", additional_kwargs={}, response_metadata={}, id='e5266ec5-bf95-4a9f-9bfd-eed3f2418fde'),
  AIMessage(content="I will send an email to john@test.com with the subject 'More information required' and body 'Please call me on whatsapp'.", additional_kwargs={'id': 'afbed5a2-df0a-4b20-bfa0-fbca6d7bf54d', 'finish_reason': 'TOOL_CALL', 'tool_plan': "I will send an email to john@test.com with the subject 'More information required' and body 'Please call me on whatsapp'.", 'tool_calls': [{'id': 'send_email_tool_1cedt1d1mr1n', 'type': 'function', 'function': {'name': 'send_email_tool', 'arguments': '{"recipient":"john@test.com","subject":"More information required","body":"Please call me on whatsapp"}'}}], 'token_count': {'input_tokens': 1541.0, 'output_tokens': 91.0}}, response_metadata={'id': 'afbed5a2-df0a-4b20-bfa0-fbca6d7bf54d', 'finish_reason': 'TOOL

In [28]:
# Step 2: Approve
from langgraph.types import Command

if "__interrupt__" in response:
    print("Approving...")

    result = agent.invoke(
        Command(resume={"decisions":[{"type":"approve"}]}),
        config=config
    )

    print(f"Result: {response['messages'][-1].content}")

Approving...
Result: I will send an email to john@test.com with the subject 'More information required' and body 'Please call me on whatsapp'.
